In [1]:
import pandas as pd
import numpy as np

import sys
import os
sys.path.append(os.path.abspath(".."))
from bblr.real_data_predictions import datasets_parameters

In [ ]:
# dataset_names = ['abalone', 'auto_mpg', 'boston', 'Brazilian_houses', 'california', 'concrete', 'diamonds', 'elevators']
# dataset_names = ['abalone', 'auto_mpg', 'boston', 'Brazilian_houses', 'concrete', 'diamonds', 'elevators']
# dataset_names = ['abalone', 'auto_mpg', 'boston', 'Brazilian_houses', 'concrete']

dataset_names = list(datasets_parameters.keys())
d_add = 0
bblr_regr_type = 'WLS'
bblr_rmses = []
lowess_rmses = []
dataset_names_found = []
bblr_found = True
dataset_names_to_remove = ['cbm']
for dataset_name in dataset_names_to_remove:
    if dataset_name in dataset_names_found:
        dataset_names_found.remove(dataset_name)
    if dataset_name in dataset_names:
        dataset_names.remove(dataset_name)

for dataset_name in dataset_names:
    n_name = str(5000) if dataset_name in ['Brazilian_houses', 'california'] else 'full'
    if n_name == 'full':
        n_name = str(5000) if dataset_name in ['house_16H', 'house_sales'] else 'full'
    # bblr_file_name = f"automl_preds/bblr_rmse_{dataset_name}_{d_add}_{n_name}.txt"
    bblr_file_name = f"automl_preds/bblr_rmse_{dataset_name}_{d_add}_{n_name}_{bblr_regr_type}.txt"
    try:
        with open(bblr_file_name, "r") as f:
            bblr_rmse = float(f.read())
        bblr_rmses.append(bblr_rmse)
        dataset_names_found.append(dataset_name)
        bblr_found = True
    except FileNotFoundError:
        print(f"File {bblr_file_name} not found. Skipping.")
        bblr_found = False
    lowess_file_name = f"automl_preds/lowess_rmse_{dataset_name}_{d_add}_{n_name}.txt"
    if bblr_found:
        try:
            with open(lowess_file_name, "r") as f:
                lowess_rmse = float(f.read())
            lowess_rmses.append(lowess_rmse)
        except FileNotFoundError:
            print(f"File {lowess_file_name} not found. Skipping.")
   
result_df = pd.DataFrame({
    'dataset': dataset_names_found,
    'bblr_rmse': bblr_rmses,
    'lowess_rmse': lowess_rmses,
    'percent_difference': [(np.round((bblr_rmses[i] - lowess_rmses[i]) / lowess_rmses[i], 4) * 100) for i in range(len(bblr_rmses))]
})

print(result_df)

File automl_preds/bblr_rmse_AmesHousing_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_california_0_5000_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_cbm_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_ccpp_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_diamonds_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_elevators_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_energy1_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_energy2_0_full_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_house_sales_0_5000_WLS.txt not found. Skipping.
File automl_preds/bblr_rmse_kin8nm_0_full_WLS.txt not found. Skipping.
            dataset     bblr_rmse   lowess_rmse  percent_difference
0           abalone      2.201834      2.132333                3.26
1  AirfoilSelfNoise      5.015824      3.011291               66.57
2          auto_mpg      3.018152      2.601475               16.02
3 

In [3]:
# comparing coefs of last four columns
dataset_name = 'AirfoilSelfNoise'
# dataset_name = 'abalone'
d_add = 4
n_name = 'full'

bblr_file_name = f"automl_preds/bblr_coefs_{dataset_name}_{d_add}_{n_name}_OLS.csv"
# bblr_file_name = f"automl_preds/bblr_coefs_{dataset_name}_{d_add}_{n_name}.csv"
lowess_file_name = f"automl_preds/lowess_coefs_{dataset_name}_{d_add}_{n_name}.csv"
bblr_coefs = pd.read_csv(bblr_file_name)
lowess_coefs = pd.read_csv(lowess_file_name)
for i in range(4):
    print(f"bias B{4-i}: BBLR = {np.round(np.mean(bblr_coefs.iloc[:, -1-i]),3)}, LOWESS = {np.round(np.mean(lowess_coefs.iloc[:, -1-i]),3)}")
    print(f"abs err B{4-i}: BBLR = {np.round(np.mean(np.abs(bblr_coefs.iloc[:, -1-i])),3)}, LOWESS = {np.round(np.mean(np.abs(lowess_coefs.iloc[:, -1-i])),3)}")
    print(f"mse B{4-i}: BBLR = {np.round(np.mean(bblr_coefs.iloc[:, -1-i]**2),3)}, LOWESS = {np.round(np.mean(lowess_coefs.iloc[:, -1-i]**2),3)}")
    print("")


bias B4: BBLR = 0.013, LOWESS = -0.027
abs err B4: BBLR = 0.566, LOWESS = 0.445
mse B4: BBLR = 0.555, LOWESS = 0.311

bias B3: BBLR = -0.218, LOWESS = -0.073
abs err B3: BBLR = 0.583, LOWESS = 0.452
mse B3: BBLR = 0.566, LOWESS = 0.309

bias B2: BBLR = -0.021, LOWESS = -0.028
abs err B2: BBLR = 0.611, LOWESS = 0.437
mse B2: BBLR = 0.607, LOWESS = 0.313

bias B1: BBLR = -0.086, LOWESS = -0.134
abs err B1: BBLR = 0.591, LOWESS = 0.481
mse B1: BBLR = 0.582, LOWESS = 0.353



In [4]:
# bias and abs error of y_pred - y_true
dataset_name = 'AirfoilSelfNoise'
dataset_name = 'auto_mpg'
dataset_name = 'Brazilian_houses'
d_add = 4
n_name = 'full'
n_name = '5000'

for dataset_name in dataset_names:
    bblr_found = False
    lowess_found = False
    for n_name in ['full', '5000']:
        try:
            bblr_file_name = f"automl_preds/bblr_predictions_{dataset_name}_{d_add}_{n_name}_OLS.csv"
            bblr_preds = pd.read_csv(bblr_file_name)
            bblr_found = True
            # bblr_file_name_found = bblr_file_name
        except FileNotFoundError:
            for n_name in ['full', '5000']:
                try:
                    bblr_file_name = f"automl_preds/bblr_predictions_{dataset_name}_{d_add}_{n_name}.csv"
                    bblr_preds = pd.read_csv(bblr_file_name)
                    bblr_found = True
                    # bblr_file_name_found = bblr_file_name
                except FileNotFoundError:
                    # print(f"File {bblr_file_name} not found. Skipping.")
                    pass
    for n_name in ['full', '5000']:
        try:
            lowess_file_name = f"automl_preds/lowess_predictions_{dataset_name}_{d_add}_{n_name}.csv"
            lowess_preds = pd.read_csv(lowess_file_name)
            lowess_found = True
            # lowess_file_name_found = lowess_file_name
        except FileNotFoundError:
            pass
            # print(f"File {lowess_file_name} not found. Skipping.")
    if bblr_found and lowess_found:
        bblr_bias = np.round(np.mean(bblr_preds['y_pred'] - bblr_preds['y_true']), 3)
        lowess_bias = np.round(np.mean(lowess_preds['y_pred'] - lowess_preds['y_true']), 3)
        bblr_abs_error = np.round(np.mean(np.abs(bblr_preds['y_pred'] - bblr_preds['y_true'])), 3)
        lowess_abs_error = np.round(np.mean(np.abs(lowess_preds['y_pred'] - lowess_preds['y_true'])), 3)
        print(f"Dataset: {dataset_name}")
        print(f"Bias: BBLR = {bblr_bias}, LOWESS = {lowess_bias}")
        print(f"Mean Absolute Error: BBLR = {bblr_abs_error}, LOWESS = {lowess_abs_error}")
        print("")

Dataset: abalone
Bias: BBLR = 0.051, LOWESS = -0.017
Mean Absolute Error: BBLR = 1.555, LOWESS = 1.516

Dataset: AirfoilSelfNoise
Bias: BBLR = 0.168, LOWESS = 0.299
Mean Absolute Error: BBLR = 3.056, LOWESS = 2.923

Dataset: auto_mpg
Bias: BBLR = 0.078, LOWESS = 0.393
Mean Absolute Error: BBLR = 2.138, LOWESS = 1.96

Dataset: boston
Bias: BBLR = -0.951, LOWESS = 0.242
Mean Absolute Error: BBLR = 2.841, LOWESS = 2.65

Dataset: Brazilian_houses
Bias: BBLR = 254.633, LOWESS = -145.599
Mean Absolute Error: BBLR = 1837.351, LOWESS = 1704.448

Dataset: brexit
Bias: BBLR = -0.3, LOWESS = -0.395
Mean Absolute Error: BBLR = 3.417, LOWESS = 2.538

Dataset: concrete
Bias: BBLR = 1.052, LOWESS = -1.79
Mean Absolute Error: BBLR = 6.457, LOWESS = 5.751

Dataset: elevators
Bias: BBLR = -0.0, LOWESS = -0.0
Mean Absolute Error: BBLR = 0.002, LOWESS = 0.002

Dataset: house_16H
Bias: BBLR = 543.764, LOWESS = 920.08
Mean Absolute Error: BBLR = 23948.196, LOWESS = 22184.949



In [7]:
# dataset_names = ['abalone', 'auto_mpg', 'boston', 'Brazilian_houses', 'california', 'concrete', 'diamonds', 'elevators']
# dataset_names = ['abalone', 'auto_mpg', 'boston', 'Brazilian_houses', 'concrete', 'diamonds', 'elevators']
# dataset_names = ['abalone', 'auto_mpg', 'boston', 'Brazilian_houses', 'concrete']

dataset_names = list(datasets_parameters.keys())
d_add = 4
# bblr_regr_type = 'WLS'
bblr_regr_types = ['OLS', 'WLS']
bblr_rmses = {dataset_key: [] for dataset_key in dataset_names}
lowess_rmses = {dataset_key: [] for dataset_key in dataset_names}
dataset_names_found = []
n_names = []
bblr_found = True

for dataset_name in dataset_names:
    n_name = str(5000) if dataset_name in ['Brazilian_houses', 'california', 'elevators'] else 'full'
    if n_name == 'full':
        n_name = str(5000) if dataset_name in ['house_16H', 'house_sales'] else 'full'
    if dataset_name in ['california', 'cbm', 'ccpp', 'kin8nm']:
        n_name = '2000'
    for bblr_regr_type in bblr_regr_types:
        try:
            bblr_file_name = f"automl_preds/bblr_rmse_{dataset_name}_{d_add}_{n_name}_{bblr_regr_type}.txt"
            with open(bblr_file_name, "r") as f:
                bblr_rmse = float(f.read())
            bblr_rmses[dataset_name].append(bblr_rmse)
            if dataset_name not in dataset_names_found:
                dataset_names_found.append(dataset_name)
            bblr_found = True
        except FileNotFoundError:
            pass
            # print(f"File {bblr_file_name} not found. Skipping.")
    if bblr_rmses[dataset_name] == []:
        bblr_found = False
    lowess_file_name = f"automl_preds/lowess_rmse_{dataset_name}_{d_add}_{n_name}.txt"
    if bblr_found:
        try:
            with open(lowess_file_name, "r") as f:
                lowess_rmse = float(f.read())
            lowess_rmses[dataset_name].append(lowess_rmse)
            n_names.append(n_name)
        except FileNotFoundError:
            print(f"File {lowess_file_name} not found. Skipping.")

bblr_rmses_found = [np.min(np.array(bblr_rmses[dataset_name])) for dataset_name in dataset_names_found]
lowess_rmses_found = [lowess_rmses[dataset_name][0] for dataset_name in dataset_names_found]

result_df = pd.DataFrame({
    'dataset': dataset_names_found,
    'n': n_names,
    'bblr_rmse': bblr_rmses_found,
    'lowess_rmse': lowess_rmses_found,
    'percent_difference': [(np.round((bblr_rmses_found[i] - lowess_rmses_found[i]) / lowess_rmses_found[i], 4) * 100) for i in range(len(bblr_rmses_found))]
})

print(result_df)

             dataset     n     bblr_rmse   lowess_rmse  percent_difference
0            abalone  full      2.153021      2.125675                1.29
1   AirfoilSelfNoise  full      4.107188      3.791860                8.32
2           auto_mpg  full      2.882649      2.701207                6.72
3             boston  full      4.108675      4.012359                2.40
4   Brazilian_houses  5000   2949.395227   2715.620947                8.61
5             brexit  full      4.254811      3.341430               27.34
6         california  2000  62720.572839  62983.056084               -0.42
7               ccpp  2000      4.428167      4.482762               -1.22
8          elevators  5000      0.002311      0.002288                1.04
9          house_16H  5000  46392.261304  43906.419085                5.66
10            kin8nm  2000      0.158917      0.159306               -0.24


In [8]:
result_df

,dataset,n,bblr_rmse,lowess_rmse,percent_difference
0,abalone,full,2.153021,2.125675,1.29
1,AirfoilSelfNoise,full,4.107188,3.791860,8.32
2,auto_mpg,full,2.882649,2.701207,6.72
3,boston,full,4.108675,4.012359,2.40
4,Brazilian_houses,5000,2949.395227,2715.620947,8.61
5,brexit,full,4.254811,3.341430,27.34
6,california,2000,62720.572839,62983.056084,-0.42
7,ccpp,2000,4.428167,4.482762,-1.22
8,elevators,5000,0.002311,0.002288,1.04
9,house_16H,5000,46392.261304,43906.419085,5.66


In [9]:
col_names = ['Dataset', 'n', 'MSE BBLR', 'MSE LOWESS', 'percent difference']
result_df.columns = col_names
my_df_str = result_df.to_latex(caption="tbd", label=f"tab:real_data_rmse_table_{d_add}", float_format="%.3f")

table_file_name = f'tables/real_data_rmse_table_{d_add}.txt'

f = open(table_file_name, "w")
with open(table_file_name, "w") as f:
    f.write(my_df_str)